# HNSW Tiering: XGBoost Training (GPU-enabled)

Train an XGBoost classifier for HNSW tiering using train/val artifacts.
- Inputs: upload `train_hnsw.pkl`, `val_hnsw.pkl` (from `scripts.hnsw_build_dataset`)
- Outputs: `model_hnsw.json`, `threshold_hnsw.json`, `metrics_hnsw.json`
- GPU: set `device='cuda'` if a GPU is available (Kaggle P100).

In [ ]:
!pip install -q --no-deps "search_system @ git+https://github.com/timothycao/search-system.git"

# Clone the repository (for configs if needed)
import os, sys, subprocess
REPO_URL = "https://github.com/timothycao/search-systems.git"
REPO_DIR = "/kaggle/working/search-systems"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
sys.path.append(REPO_DIR)

In [ ]:
import json
from pathlib import Path
import numpy as np
import xgboost as xgb
from systems.tiering import load_dataset, select_threshold, evaluate_at_threshold

# Configure paths
INPUT_DATASET = Path("/kaggle/input/hnsw-tiering")  # adjust to your dataset name
TRAIN_PATH = INPUT_DATASET / "train_hnsw.pkl"
VAL_PATH = INPUT_DATASET / "val_hnsw.pkl"

OUTPUT_DIR = Path("/kaggle/working/artifacts/tiering_dense")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUTPUT_DIR / "model_hnsw.json"
METRICS_PATH = OUTPUT_DIR / "metrics_hnsw.json"
THRESHOLD_PATH = OUTPUT_DIR / "threshold_hnsw.json"

# Hyperparameters
TARGET_RATIO = 0.4
NUM_ROUNDS = 500
EARLY_STOPPING = 50
LEARNING_RATE = 0.05
MAX_DEPTH = 6
SUBSAMPLE = 0.8
COLSAMPLE_BYTREE = 0.8
SEED = 42
DEVICE = "cuda"  # set to "cpu" if no GPU

# Load datasets
train_ds = load_dataset(str(TRAIN_PATH))
val_ds = load_dataset(str(VAL_PATH))

dtrain = xgb.DMatrix(train_ds["X"], label=train_ds["y"], feature_names=train_ds["feature_names"])
dval = xgb.DMatrix(val_ds["X"], label=val_ds["y"], feature_names=val_ds["feature_names"])
evals = [(dtrain, "train"), (dval, "val")]

params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "eta": LEARNING_RATE,
    "max_depth": MAX_DEPTH,
    "subsample": SUBSAMPLE,
    "colsample_bytree": COLSAMPLE_BYTREE,
    "tree_method": "hist",
    "device": DEVICE,
    "seed": SEED,
}

booster = xgb.train(
    params,
    dtrain,
    num_boost_round=NUM_ROUNDS,
    evals=evals,
    early_stopping_rounds=EARLY_STOPPING,
    verbose_eval=50,
)

# Threshold selection
val_probs = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1))
threshold = select_threshold(list(val_probs), target_ratio=TARGET_RATIO)
pr_metrics = evaluate_at_threshold(list(val_probs), val_ds["y"], threshold)

# Save outputs
booster.save_model(MODEL_PATH)

def to_py(o):
    if isinstance(o, dict):
        return {k: to_py(v) for k, v in o.items()}
    if isinstance(o, list):
        return [to_py(v) for v in o]
    if isinstance(o, np.generic):
        return o.item()
    return o

metrics_out = to_py({
    "training": {
        "best_iteration": booster.best_iteration,
        "best_score": booster.best_score,
        "eval_metric": params["eval_metric"],
    },
    "threshold": float(threshold),
    "target_ratio": float(TARGET_RATIO),
    "val_precision": float(pr_metrics["precision"]),
    "val_recall": float(pr_metrics["recall"]),
    "val_pred_ratio": float(pr_metrics["pred_ratio"]),
})

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_out, f, indent=2)

with open(THRESHOLD_PATH, "w", encoding="utf-8") as f:
    json.dump({"threshold": float(threshold), "target_ratio": float(TARGET_RATIO)}, f, indent=2)

metrics_out
